# 04_2 — FAISS Vector Index (Chunk-Level)

Build a FAISS index over chunk embeddings for fast exact cosine similarity search.

**Why FAISS:** Pure local, no server needed, zero network latency. `IndexFlatIP` with
L2-normalized vectors gives exact cosine similarity. For < 100k chunks this is plenty fast.

**Inputs** (from notebooks 02 & 03):
- `data/processed/chunk_embeddings.npy` — (N_chunks, 384) float32
- `data/processed/chunks_with_embeddings.json` — chunk metadata aligned by row index
- `data/processed/resume_chunks.json` — full chunk text (for result previews)

**Outputs** (consumed by the Streamlit app):
- `resume_index.faiss` — FAISS index (DS3 member chunks only)
- `member_chunks_metadata.json` — chunk metadata aligned to the FAISS index
- `config.json` — model name, dimensions, counts

**Comparison with 04_1 (Qdrant):** FAISS is lower-level but faster for pure ANN.
Qdrant adds metadata filtering before search; FAISS filters post-hoc.

In [1]:
import json
import numpy as np
from pathlib import Path
from collections import Counter
from sentence_transformers import SentenceTransformer
import faiss

PROJECT_ROOT  = Path("../").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EMB_PATH      = PROCESSED_DIR / "chunk_embeddings.npy"
META_PATH     = PROCESSED_DIR / "chunks_with_embeddings.json"
CHUNKS_PATH   = PROCESSED_DIR / "resume_chunks.json"

# Final artifacts (paths match streamlit/config.py)
FAISS_OUT     = PROJECT_ROOT / "resume_index.faiss"
METADATA_OUT  = PROJECT_ROOT / "member_chunks_metadata.json"
CONFIG_OUT    = PROJECT_ROOT / "config.json"

MODEL_NAME    = "all-MiniLM-L6-v2"

print(f"Embeddings : {EMB_PATH}")
print(f"Chunk meta : {META_PATH}")
print(f"FAISS out  : {FAISS_OUT}")

Embeddings : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/chunk_embeddings.npy
Chunk meta : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/chunks_with_embeddings.json
FAISS out  : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/resume_index.faiss


## 1 — Load chunk embeddings & metadata

In [2]:
all_embeddings = np.load(EMB_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    all_meta = json.load(f)

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

assert len(all_embeddings) == len(all_meta) == len(all_chunks), (
    f"Row count mismatch: embeddings={len(all_embeddings)}, "
    f"meta={len(all_meta)}, chunks={len(all_chunks)}"
)

print(f"Loaded {len(all_embeddings)} chunk embeddings  (dim={all_embeddings.shape[1]})")

section_counts = Counter(m["section_type"] for m in all_meta)
print(f"\n{'Section':<20} {'Chunks':>8}")
print("-" * 30)
for sec, cnt in sorted(section_counts.items(), key=lambda x: -x[1]):
    print(f"{sec:<20} {cnt:>8}")

source_counts = Counter(m["source"] for m in all_meta)
print(f"\n{'Source':<20} {'Chunks':>8}")
print("-" * 30)
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"{src:<20} {cnt:>8}")

Loaded 60405 chunk embeddings  (dim=384)

Section                Chunks
------------------------------
experience              38691
education               11352
projects                 2709
contact                  2298
skills                   2164
summary                  1546
certifications            639
awards                    440
volunteer                 236
publications              193
interests                  91
languages                  24
references                 22

Source                 Chunks
------------------------------
train                   58741
ds3_members              1420
ds3_board                 244


## 2 — Filter to DS3 member chunks (production index)

The production index only indexes DS3 member chunks — those are the candidates
being searched. Other sources (kaggle, train) are for training / evaluation only.

In [3]:
member_indices = [i for i, m in enumerate(all_meta) if m["source"] in ("ds3_members", "ds3_board")]

member_embeddings = all_embeddings[member_indices].copy().astype("float32")
member_meta       = [all_meta[i] for i in member_indices]
member_chunks     = [all_chunks[i] for i in member_indices]

print(f"DS3 chunks selected for index: {len(member_indices)}")
sec_dist = Counter(m["section_type"] for m in member_meta)
for sec, cnt in sorted(sec_dist.items(), key=lambda x: -x[1]):
    print(f"  {sec:<20} {cnt}")

if len(member_indices) == 0:
    print("\n[WARNING] No DS3 chunks found — falling back to ALL chunks.")
    member_embeddings = all_embeddings.copy().astype("float32")
    member_meta       = all_meta
    member_chunks     = all_chunks

DS3 chunks selected for index: 1664
  education            314
  contact              312
  skills               266
  experience           259
  projects             226
  volunteer            109
  publications         63
  awards               50
  summary              37
  certifications       19
  languages            6
  interests            3


## 3 — Build the FAISS index

`IndexFlatIP` with L2-normalized vectors = exact cosine similarity search.
For < 100k chunks this runs in milliseconds and requires no training step.

In [4]:
dimension = member_embeddings.shape[1]

# Vectors must already be L2-normalized (they are, from notebook 03).
# normalize_L2 is idempotent but harmless to call again as a safety net.
faiss.normalize_L2(member_embeddings)

index = faiss.IndexFlatIP(dimension)
index.add(member_embeddings)

print(f"FAISS index built")
print(f"  Index type : IndexFlatIP (exact cosine similarity)")
print(f"  Vectors    : {index.ntotal}")
print(f"  Dimension  : {dimension}")

FAISS index built
  Index type : IndexFlatIP (exact cosine similarity)
  Vectors    : 1664
  Dimension  : 384


## 4 — Test search

Chunk-level search returns individual evidence snippets. We then group the top
chunks by `candidate_id` and score candidates by their best-matching chunk
(max pooling) or average of top-k chunks per candidate.

In [5]:
model = SentenceTransformer(MODEL_NAME)


def search_chunks(query: str, top_k: int = 20, section_filter: str | None = None):
    """
    Return the top-k most similar chunks to *query*.
    Optionally restrict to a specific section_type (e.g. 'experience', 'skills').
    """
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k * 3)  # oversample before optional filter

    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx < 0:
            continue
        meta = member_meta[idx]
        if section_filter and meta["section_type"] != section_filter:
            continue
        results.append({
            "rank":         len(results) + 1,
            "score":        round(float(score), 4),
            "candidate_id": meta["candidate_id"],
            "section_type": meta["section_type"],
            "source":       meta["source"],
            "text":         member_chunks[idx]["text"][:300],
        })
        if len(results) >= top_k:
            break
    return results


def search_candidates(query: str, top_k_candidates: int = 10, chunks_per_candidate: int = 3):
    """
    Roll up chunk-level results to candidate-level using max-score pooling.
    Returns one row per candidate with their best chunk score and top matching snippets.
    """
    raw = search_chunks(query, top_k=top_k_candidates * chunks_per_candidate * 2)

    from collections import defaultdict
    candidate_hits: dict = defaultdict(list)
    for hit in raw:
        candidate_hits[hit["candidate_id"]].append(hit)

    candidates = []
    for cid, hits in candidate_hits.items():
        hits_sorted = sorted(hits, key=lambda h: -h["score"])
        candidates.append({
            "candidate_id": cid,
            "best_score":   hits_sorted[0]["score"],
            "top_chunks":   hits_sorted[:chunks_per_candidate],
        })

    candidates.sort(key=lambda c: -c["best_score"])
    return candidates[:top_k_candidates]


print("Search functions ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Search functions ready.


In [6]:
test_queries = [
    ("Python machine learning data science", None),
    ("React JavaScript full-stack web development", None),
    ("C++ computer vision embedded systems", None),
    ("built APIs with Python", "experience"),         # section-filtered search
    ("SQL database analytics", "skills"),             # skills-only search
]

for query, section_filter in test_queries:
    label = f"[{section_filter}] " if section_filter else ""
    print(f"\n{'=' * 70}")
    print(f"Query: {label}{query}")
    print(f"{'=' * 70}")
    results = search_chunks(query, top_k=5, section_filter=section_filter)
    for r in results:
        print(f"  #{r['rank']}  score={r['score']:.4f}  [{r['section_type']}]  {r['candidate_id']}")
        print(f"       {r['text'][:120]}")


Query: Python machine learning data science


  #1  score=0.6485  [education]  board_resume_44.pdf
       University of California, San Diego | Cumulative GPA 3.94 La Jolla, CA
Bachelor of Science: Data Science | Machine Learn
  #2  score=0.6221  [skills]  member_resume_joseph_rodriguez.pdf
       Python (PyTorch, Tensorflow, Scikit, learn, Pandas
  #3  score=0.6028  [projects]  member_resume_bala_vazrala.pdf
       Comparative Machine Learning Analysis | UCI ML Repository January 2026 - Present
Collaborating in a four-person team to 
  #4  score=0.5900  [skills]  member_resume_yra_climaco.pdf
       Machine Learning & Statistics: Scikit, learn, Regression, Classification, Survival Analysis, Ensemble Methods, Data Clea
  #5  score=0.5892  [skills]  member_resume_nehal_choudhary.pdf
       ML & Data: pandas, NumPy, scikit, learn, FAISS; regression, classification, time series, experimentation, Visualization 

Query: React JavaScript full-stack web development


  #1  score=0.4642  [projects]  member_resume_anthony_wang.pdf
       BoilerAlgorithms | Python, TypeScript, Django, React.js, HTML/CSS, MongoDB
  #2  score=0.4520  [skills]  member_resume_william_ayoade.pdf
       Python  Javascript, Java  C++, C  React, HTML/CSS  Excel
  #3  score=0.4416  [skills]  member_resume_david_wan.pdf
       Typescript, HTML, CSS, Python, Java, Node, js, React.js, Data Structures and Algorithms, Google Sheets, MATLAB
  #4  score=0.4402  [skills]  member_resume_jamera_fernando.pdf
       Java, Python, JavaScript, CSS, React, React Native, PostgreSQL, MongoDB, Firebase, AWS (Lambda, S3), TensorFlow, Pytorch
  #5  score=0.4348  [skills]  member_resume_alexander_hughes.pdf
       React, Node.js, WordPress, Git, Docker, VS Code, Visual Studio, pandas, NumPy, Matplotlib

Query: C++ computer vision embedded systems
  #1  score=0.4337  [skills]  member_resume_pali_jain.pdf
       Python and C++ Object, oriented programming Model optimization
  #2  score=0.4227  [skil

## 5 — Candidate-level rollup

Show top candidates for a query by rolling up their best chunk scores.

In [7]:
query = "Python machine learning data science model deployment"
print(f"Query: {query}\n")

candidates = search_candidates(query, top_k_candidates=5)
for c in candidates:
    print(f"  Candidate : {c['candidate_id']}  (best_score={c['best_score']:.4f})")
    for chunk in c["top_chunks"]:
        print(f"    [{chunk['section_type']}] score={chunk['score']:.4f}  {chunk['text'][:100]}")
    print()

Query: Python machine learning data science model deployment

  Candidate : member_resume_rishab_kamalesh.pdf  (best_score=0.6017)
    [skills] score=0.6017  Data Pipelines, Experimentation Infrastructure, Data & ML: Pandas, NumPy, PyTorch, Scikit, Learn, Fe

  Candidate : member_resume_david_yu.pdf  (best_score=0.5646)
    [summary] score=0.5646  Data Science student and ML enthusiast with production experience building backends and shipping AI 

  Candidate : board_resume_20.pdf  (best_score=0.5220)
    [summary] score=0.5220  Second-year Data Science student with strong experience in Python, SQL, and Git, and a solid foundat

  Candidate : member_resume_kaii_bijlani.pdf  (best_score=0.5215)
    [projects] score=0.5215  Machine Learning Under the Hood | Python, Numpy, FastAPI, TypeScript, MongoDB, React September 2025 

  Candidate : member_resume_aidan_tjon.pdf  (best_score=0.5198)
    [education] score=0.5198  University of California San Diego Class of 2029
Code Hobbits, Fremont J

## 6 — Export artifacts for Streamlit

In [8]:
# 1. FAISS index
faiss.write_index(index, str(FAISS_OUT))
print(f"Saved FAISS index  : {FAISS_OUT}  ({FAISS_OUT.stat().st_size / 1024:.1f} KB)")

# 2. Chunk metadata — includes full text for UI previews
export_meta = []
for meta, chunk in zip(member_meta, member_chunks):
    export_meta.append({
        "chunk_id":     meta["chunk_id"],
        "candidate_id": meta["candidate_id"],
        "source":       meta["source"],
        "section_type": meta["section_type"],
        "text":         chunk["text"],
        "metadata":     meta.get("metadata", {}),
    })

with open(METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(export_meta, f, indent=2, ensure_ascii=False)
print(f"Saved chunk metadata: {METADATA_OUT}  ({METADATA_OUT.stat().st_size / 1024:.1f} KB)")

# 3. Config
config = {
    "model_name":          MODEL_NAME,
    "embedding_dimension": dimension,
    "num_chunks":          index.ntotal,
    "index_type":          "IndexFlatIP",
    "index_backend":       "faiss",
}
with open(CONFIG_OUT, "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved config        : {CONFIG_OUT}")

print(f"\nDone! Streamlit app can load resume_index.faiss + member_chunks_metadata.json")

Saved FAISS index  : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/resume_index.faiss  (2496.0 KB)
Saved chunk metadata: /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/member_chunks_metadata.json  (1142.7 KB)
Saved config        : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/config.json

Done! Streamlit app can load resume_index.faiss + member_chunks_metadata.json
